In [ ]:
from pathlib import Path
import av
import numpy as np
import torch
import polars as pl

# Config
data_path = "../data/video_data.parquet"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load metadata
df = pl.read_parquet(data_path)

df = df.filter(
    pl.col('media_type') == 'video',
    pl.col('channel_handle') != '@SecretBaseSBN',
    pl.col("view_count").is_not_null(),
    )

df.shape

In [ ]:
# Get unique channel list
channels = df.select("channel_id").unique().to_series()

# List to collect per-channel normalized DataFrames
normalized_chunks = []

# Loop through channels and normalize
for ch in channels:
    
    channel_df = df.filter(pl.col("channel_id") == ch)
    # log scale view_count
    channel_df = channel_df.with_columns(
        pl.col("view_count").log().alias("log_view_count")
    )

    # set target to log view_count
    channel_df = channel_df.with_columns(
        pl.col("log_view_count").alias("target")
    )
    
    # Normalize view count by Z-score
    mean_vc = channel_df.select(pl.col("log_view_count").mean()).item()
    std_vc = channel_df.select(pl.col("log_view_count").std()).item()
    channel_df = channel_df.with_columns(
        ((pl.col("log_view_count") - mean_vc) / std_vc).alias("target")
    )

    # Add mean, std for debug
    channel_df = channel_df.with_columns(
        pl.lit(mean_vc).alias("mean_log_view_count"),
        pl.lit(std_vc).alias("std_log_view_count")
    )

    normalized_chunks.append(channel_df)

# Concatenate all normalized chunks
norm_df = pl.concat(normalized_chunks)
norm_df = norm_df.drop_nulls(subset=["target", "video_title"])

norm_df.shape

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel
from torch.utils.data import Dataset

def collate_fn(batch):
    input_ids = torch.stack([item['input_ids'] for item in batch])
    attention_mask = torch.stack([item['attention_mask'] for item in batch])
    targets = torch.stack([item['target'] for item in batch])
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "target": targets
    }

class TitleDataset(Dataset):
    def __init__(self, df, tokenizer, target_col="target", max_length=32):
        self.targets = df[target_col].to_list()
        self.encodings = tokenizer(
            df["video_title"].to_list(),
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["target"] = torch.tensor(self.targets[idx]).float()
        return item

class TitleRegressor(nn.Module):
    def __init__(self, model_name, dropout_rate=0.2, freeze_n_layers=None):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)

        print(self.encoder.config)

        if freeze_n_layers is not None:
            self._freeze_layers(freeze_n_layers)
        
        hidden_dim = self.encoder.config.hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 1)  # Output layer for regression
        )

    def _freeze_layers(self, n):
        # Freeze embeddings if present
        if hasattr(self.encoder, 'embeddings'):
            for param in self.encoder.embeddings.parameters():
                param.requires_grad = False

        # Try locating layer stack via known patterns
        candidate_paths = [
            ("encoder.layer", getattr(self.encoder, "encoder", None)),
            ("transformer.layer", getattr(self.encoder, "transformer", None)),
            ("distilbert.transformer.layer", getattr(getattr(self.encoder, "distilbert", None), "transformer", None)),
            ("layers", self.encoder),  # ModernBERT: layers at top-level
        ]

        for name, module in candidate_paths:
            if module and hasattr(module, "layer"):
                layers = getattr(module, "layer")
                if isinstance(layers, nn.ModuleList):
                    break
            elif isinstance(module, nn.Module) and hasattr(module, "layers"):
                layers = getattr(module, "layers")
                if isinstance(layers, nn.ModuleList):
                    break
        else:
            # Fallback: search recursively for first ModuleList with enough layers
            layers = next(
                (m for m in self.encoder.modules() if isinstance(m, nn.ModuleList) and len(m) >= n),
                None
            )
            if layers is None:
                raise ValueError("Cannot locate transformer layers in the model")

        # Cap n at available number of layers
        n = min(n, len(layers))
        print(f"Freezing first {n} of {len(layers)} layers.")

        for layer in layers[:n]:
            for param in layer.parameters():
                param.requires_grad = False


    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # CLS token
        return self.regressor(cls_embedding)

    def summary(self):
        print(f"TitleRegressor with {self.encoder.config.hidden_size} hidden size and {len(self.regressor)} layers.")
        total, trainable = 0, 0
        for name, param in self.named_parameters():  # fixed from `model.named_parameters()`
            total += param.numel()
            if param.requires_grad:
                trainable += param.numel()
                print(f"Trainable: {name} - {param.numel()} params")
        print(f"\nTrainable parameters: {trainable:,} / {total:,}")

from sklearn.model_selection import train_test_split

from enum import Enum
# Model configuration
class ModelType(Enum):
    BERT_TINY = "prajjwal1/bert-tiny"
    DISTILBERT_BASE = "distilbert/distilbert-base-uncased"
    BERT_BASE_UNCASED = "google-bert/bert-base-uncased"
    MODERN_BERT_BASE = "answerdotai/ModernBERT-base"
    MPNET_BASE = "microsoft/mpnet-base"
    CONTRIEVER = "facebook/contriever"
    ALL_MINILM_L6_V2 = "sentence-transformers/all-MiniLM-L6-v2"
    E5_BASE_V2 = "intfloat/e5-base-v2"
    YOUTUBE_BERT = "flboehm/youtube-bert"
    YOUTUBE_BERT_10 = "flboehm/youtube-bert_10"
    T5_SMALL_YOUTUBE = "marianna13/t5-small-finetuned-youtube"
    DEBERTA_V3_XSMALL = "microsoft/deberta-v3-xsmall"
    YOUTUBE_XLM_ROBERTA_BASE = "AmaanP314/youtube-xlm-roberta-base-sentiment-multilingual"

model_selection = ModelType.YOUTUBE_BERT
model_name = model_selection.value
test_frac = 0.1
batch_size = 8192

# Channel-level split
channel_ids = norm_df["channel_id"].unique().to_list()
train_ch, test_ch = train_test_split(channel_ids, test_size=test_frac, random_state=42)

train_df = norm_df.filter(pl.col("channel_id").is_in(train_ch))
test_df = norm_df.filter(pl.col("channel_id").is_in(test_ch))

from transformers import AutoTokenizer
text_tokenizer = AutoTokenizer.from_pretrained(model_name)

train_dataset = TitleDataset(train_df, tokenizer=text_tokenizer)
test_dataset = TitleDataset(test_df, tokenizer=text_tokenizer)

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

model = TitleRegressor(
    model_name=model_name,
    freeze_n_layers=22,
    dropout_rate=0.5
)
    
model.to(device)
model.summary()

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = torch.nn.HuberLoss()
epochs = 256

from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

log_dir = f"runs/title_regression_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
writer = SummaryWriter(log_dir=log_dir)

from sklearn.metrics import mean_squared_error, r2_score
from tqdm.notebook import trange

for epoch in trange(epochs, desc="Training Epochs"):
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["target"].unsqueeze(1).to(device)

        preds = model(input_ids, attention_mask)
        loss = loss_fn(preds, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Log training loss
    writer.add_scalar("Loss/Train", total_loss / len(train_loader), epoch)

    if (epoch + 1) % 4 == 0:
        model.eval()
        val_preds = []
        val_targets = []
        val_loss = 0

        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                targets = batch["target"].unsqueeze(1).to(device)

                preds = model(input_ids, attention_mask)
                loss = loss_fn(preds, targets)

                val_preds.extend(preds.squeeze().cpu().tolist())
                val_targets.extend(targets.squeeze().cpu().tolist())
                val_loss += loss.item()

        val_mse = mean_squared_error(val_targets, val_preds)
        val_r2 = r2_score(val_targets, val_preds)

        # Log validation metrics
        writer.add_scalar("Loss/Val", val_loss / len(test_loader), epoch)
        writer.add_scalar("Metrics/Val_MSE", val_mse, epoch)
        writer.add_scalar("Metrics/Val_R2", val_r2, epoch)

        print(f"Epoch {epoch+1}/{epochs} - "
              f"Train Loss: {total_loss / len(train_loader):.4f} | "
              f"Val Loss: {val_loss / len(test_loader):.4f} | "
              f"Val MSE: {val_mse:.4f} | Val R2: {val_r2:.4f}")
